# Day 10: End-to-End Orchestration, Observability & Verification

This notebook simulates a complete E2E run of a synthetic patient from raw HL7 JSON -> Bronze -> Tokenization -> Silver OMOP -> DQ -> Gold readmission metric.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ -q

In [ ]:
import os
import json
import hashlib
import duckdb
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col, udf, to_date, date_add
from pyspark.sql.types import StringType, IntegerType
import pydeequ
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite

# Setup environment
os.environ["EHDIP_HASH_SALT"] = "e2e_secret_salt"
os.environ["SPARK_VERSION"] = "3.5"

spark = SparkSession.builder \
    .appName("Day10_E2E_Walkthrough") \
    .config("spark.jars.packages", "com.amazon.deequ:deequ:2.0.7-spark-3.3,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/e2e_iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Ready for E2E Pipeline.")

In [ ]:
### 1. Ingestion (Raw -> Bronze)
raw_hl7_json = [
    '{"msg_id": "M1", "patient_id": "PAT_A", "ssn": "123-45-6789", "dob": "1985-01-01", "enc_id": "E1", "enc_type": "inpatient", "start": "2023-11-01", "end": "2023-11-05"}',
    '{"msg_id": "M2", "patient_id": "PAT_A", "ssn": "123-45-6789", "dob": "1985-01-01", "enc_id": "E2", "enc_type": "inpatient", "start": "2023-11-15", "end": null}' # Readmission
]

df_raw = spark.createDataFrame([(j,) for j in raw_hl7_json], ["raw_payload_json"])

bronze_df = df_raw.select(
    current_timestamp().alias("ingestion_timestamp"),
    lit("HL7_SYSTEM").alias("source_system_id"),
    col("raw_payload_json")
)

spark.sql("CREATE DATABASE IF NOT EXISTS local.e2e_db")
bronze_df.writeTo("local.e2e_db.bronze").createOrReplace()
print("1. Ingestion Complete. Data in Bronze.")

In [ ]:
### 2. De-Identification (Bronze -> Bronze De-ID)
df_bronze = spark.table("local.e2e_db.bronze")

# Extract JSON fields for processing
df_parsed = spark.read.json(df_bronze.rdd.map(lambda row: row.raw_payload_json))

def get_salt(): return os.environ["EHDIP_HASH_SALT"]
fpe_udf = udf(lambda v: hashlib.sha256((str(v)+get_salt()).encode()).hexdigest()[:16] if v else v, StringType())
shift_udf = udf(lambda pid: (int(hashlib.sha256((str(pid)+get_salt()).encode()).hexdigest()[:8], 16) % 61) - 30 if pid else 0, IntegerType())

df_deid = df_parsed.withColumn("shift", shift_udf(col("patient_id"))) \
                   .withColumn("patient_id_masked", fpe_udf(col("patient_id"))) \
                   .withColumn("ssn_masked", fpe_udf(col("ssn"))) \
                   .withColumn("dob_shifted", date_add(to_date(col("dob")), col("shift"))) \
                   .drop("ssn", "dob", "shift")

print("2. De-Identification Complete. PHI secured.")

In [ ]:
### 3. OMOP Transformation (Bronze De-ID -> Silver)
omop_df = df_deid.select(
    col("enc_id").alias("condition_occurrence_id"),
    col("patient_id_masked").alias("person_id"),
    lit(32020).alias("condition_type_concept_id"), # EHR Encounter
    to_date(col("start")).alias("condition_start_date"),
    to_date(col("end")).alias("condition_end_date")
)
omop_df.writeTo("local.e2e_db.silver_omop").createOrReplace()
print("3. OMOP Transformation Complete. Data in Silver format.")

In [ ]:
### 4. Data Quality Check
df_silver = spark.table("local.e2e_db.silver_omop")
check = Check(spark, CheckLevel.Error, "OMOP_DQ")
res = VerificationSuite(spark).onData(df_silver) \
    .addCheck(check.isComplete("person_id").isComplete("condition_occurrence_id")) \
    .run()

print("4. Data Quality Check Results:")
VerificationSuite.checkResultsAsDataFrame(spark, res).show(truncate=False)

In [ ]:
### 5. Gold Mart Simulation (Readmissions)
# Export to pandas for DuckDB simulation
pdf_silver = df_silver.toPandas()
pdf_silver['condition_start_date'] = pd.to_datetime(pdf_silver['condition_start_date'])
pdf_silver['condition_end_date'] = pd.to_datetime(pdf_silver['condition_end_date'])

conn = duckdb.connect(':memory:')
conn.register('silver_omop', pdf_silver)

query = """
WITH discharges AS (
    SELECT person_id, condition_occurrence_id AS d_enc, condition_end_date AS d_date
    FROM silver_omop WHERE condition_end_date IS NOT NULL
),
admissions AS (
    SELECT person_id, condition_occurrence_id AS a_enc, condition_start_date AS a_date
    FROM silver_omop
)
SELECT
    d.person_id,
    d.d_enc, d.d_date,
    a.a_enc, a.a_date,
    DATE_DIFF('day', CAST(d.d_date AS TIMESTAMP), CAST(a.a_date AS TIMESTAMP)) AS days_to_readmission
FROM discharges d
JOIN admissions a ON d.person_id = a.person_id
 AND a.a_date > d.d_date
 AND DATE_DIFF('day', CAST(d.d_date AS TIMESTAMP), CAST(a.a_date AS TIMESTAMP)) <= 30
"""
print("5. Gold Mart Execution: 30-Day Readmissions Found!")
print(conn.execute(query).df())